# 🎮 Notebook 05 — Coba Prediksi Sendiri

---

Notebook ini khusus untuk **mencoba model secara interaktif**.

Kamu tinggal **ketik nama dua tim**, lalu model akan:
1. Memprediksi siapa yang menang (atau seri)
2. Menjelaskan **alasan** kenapa prediksinya seperti itu
3. Menampilkan probabilitas tiap kemungkinan hasil

Cocok dipakai langsung di **Google Colab** tanpa perlu install apapun.

### Setup untuk Google Colab

Jalankan cell di bawah ini dulu jika kamu membuka notebook ini lewat **Google Colab**.

Kalau dijalankan secara lokal (Jupyter Notebook di laptop), cell ini otomatis dilewati.

In [ ]:
import os

IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    if not os.path.exists('worldcup-win-predicition'):
        get_ipython().system('git clone https://github.com/vnymyz/worldcup-win-predicition.git')
    os.chdir('worldcup-win-predicition/notebooks')
    print('Setup Colab selesai! Folder kerja:', os.getcwd())
else:
    print('Berjalan secara lokal -- tidak perlu clone repo.')

## 1. Import Library & Load Model

In [1]:
import pandas as pd
import pickle
import difflib

with open('../models/model.pkl', 'rb') as f:
    payload = pickle.load(f)

model         = payload['model']
model_name    = payload['model_name']
features      = payload['features']
win_rate_dict = payload['win_rate_dict']
accuracy      = payload['accuracy']

teams = sorted(win_rate_dict.keys())

print(f'Model dimuat: {model_name} (akurasi {accuracy*100:.1f}%)')
print(f'Total {len(teams)} tim tersedia.')

Model dimuat: Logistic Regression (akurasi 56.1%)
Total 84 tim tersedia.


## 2. Lihat Daftar Nama Tim

Pastikan kamu mengetik nama tim **sesuai daftar ini** (case-sensitive, contoh: `Brazil`, bukan `brazil`).

In [2]:
print(', '.join(teams))

Algeria, Angola, Argentina, Australia, Austria, Belgium, Bolivia, Bosnia and Herzegovina, Brazil, Bulgaria, Cameroon, Canada, Cape Verde, Chile, China, Colombia, Costa Rica, Croatia, Cuba, Curaçao, Czech Republic, Czechoslovakia, DR Congo, Denmark, Ecuador, Egypt, El Salvador, England, France, German DR, Germany, Ghana, Greece, Haiti, Honduras, Hungary, Iceland, Indonesia, Iran, Iraq, Israel, Italy, Ivory Coast, Jamaica, Japan, Kuwait, Mexico, Morocco, Netherlands, New Zealand, Nigeria, North Korea, Northern Ireland, Norway, Panama, Paraguay, Peru, Poland, Portugal, Qatar, Republic of Ireland, Romania, Russia, Saudi Arabia, Scotland, Senegal, Serbia, Slovakia, Slovenia, South Africa, South Korea, Spain, Sweden, Switzerland, Togo, Trinidad and Tobago, Tunisia, Turkey, Ukraine, United Arab Emirates, United States, Uruguay, Wales, Yugoslavia


## 3. Fungsi Prediksi + Penjelasan

Fungsi ini akan:
- Mencocokkan nama tim yang kamu ketik (otomatis koreksi typo kecil)
- Menghitung prediksi hasil
- Menjelaskan **kenapa** model memprediksi seperti itu

In [3]:
def cari_nama_tim(nama_input):
    """Cari nama tim yang paling cocok, toleran typo kecil."""
    if nama_input in teams:
        return nama_input
    cocok = difflib.get_close_matches(nama_input, teams, n=1, cutoff=0.6)
    return cocok[0] if cocok else None


def prediksi_dan_jelaskan(tim_a_input, tim_b_input, lapangan_netral=True):
    tim_a = cari_nama_tim(tim_a_input)
    tim_b = cari_nama_tim(tim_b_input)

    if tim_a is None:
        print(f'Nama tim "{tim_a_input}" tidak ditemukan di data. Cek daftar tim di atas.')
        return
    if tim_b is None:
        print(f'Nama tim "{tim_b_input}" tidak ditemukan di data. Cek daftar tim di atas.')
        return
    if tim_a == tim_b:
        print('Pilih dua tim yang berbeda ya!')
        return

    wr_a = win_rate_dict.get(tim_a, 0.33)
    wr_b = win_rate_dict.get(tim_b, 0.33)
    diff = wr_a - wr_b
    netral = 1 if lapangan_netral else 0

    fitur = pd.DataFrame([[wr_a, wr_b, diff, netral]], columns=features)
    hasil = model.predict(fitur)[0]
    proba = model.predict_proba(fitur)[0]
    kelas = model.classes_

    print('=' * 55)
    print(f'  {tim_a}  vs  {tim_b}')
    print('=' * 55)

    if hasil == 'W':
        print(f'PREDIKSI: {tim_a} MENANG')
    elif hasil == 'L':
        print(f'PREDIKSI: {tim_b} MENANG')
    else:
        print('PREDIKSI: SERI')

    print()
    print('--- Alasan ---')
    selisih_persen = abs(diff) * 100
    if hasil == 'W':
        print(f'{tim_a} punya win rate historis {wr_a*100:.1f}%, lebih tinggi '
              f'{selisih_persen:.1f} poin dibanding {tim_b} ({wr_b*100:.1f}%).')
        print('Tim dengan riwayat kemenangan lebih banyak di World Cup cenderung diprediksi menang lagi.')
    elif hasil == 'L':
        print(f'{tim_b} punya win rate historis {wr_b*100:.1f}%, lebih tinggi '
              f'{selisih_persen:.1f} poin dibanding {tim_a} ({wr_a*100:.1f}%).')
        print('Tim dengan riwayat kemenangan lebih banyak di World Cup cenderung diprediksi menang.')
    else:
        print(f'Win rate kedua tim cukup berimbang -- {tim_a} {wr_a*100:.1f}% vs '
              f'{tim_b} {wr_b*100:.1f}% (selisih hanya {selisih_persen:.1f} poin).')
        print('Karena kekuatan historisnya mirip, model memprediksi hasil seri.')

    if netral == 0:
        print(f'Faktor tambahan: {tim_a} bermain di kandang sendiri, memberi sedikit keuntungan ekstra.')

    print()
    print('--- Probabilitas ---')
    label_tampil = {'W': f'{tim_a} Menang', 'D': 'Seri', 'L': f'{tim_b} Menang'}
    urut = sorted(zip(kelas, proba), key=lambda x: -x[1])
    for k, p in urut:
        print(f'{label_tampil[k]:<25} {p*100:.1f}%')
    print('=' * 55)

## 4. Coba Sekarang!

Jalankan cell di bawah ini, lalu **ketik nama dua tim** saat diminta.

In [4]:
tim_a_input = input('Masukkan Tim A (Home): ')
tim_b_input = input('Masukkan Tim B (Away): ')
netral_input = input('Lapangan netral? (y/n, default y): ').strip().lower()

lapangan_netral = (netral_input != 'n')

print()
prediksi_dan_jelaskan(tim_a_input, tim_b_input, lapangan_netral)


  Indonesia  vs  Japan
PREDIKSI: Japan MENANG

--- Alasan ---
Japan punya win rate historis 26.9%, lebih tinggi 26.9 poin dibanding Indonesia (0.0%).
Tim dengan riwayat kemenangan lebih banyak di World Cup cenderung diprediksi menang.

--- Probabilitas ---
Japan Menang              57.6%
Seri                      28.5%
Indonesia Menang          13.9%


## 5. Mau Coba Lagi?

Jalankan ulang cell di atas (cell nomor 4) untuk mencoba pasangan tim yang berbeda.

Atau panggil langsung fungsinya tanpa perlu mengetik input satu-satu:

In [5]:
# Contoh panggil langsung tanpa input manual
prediksi_dan_jelaskan('Brazil', 'Germany')
print()
prediksi_dan_jelaskan('Argentina', 'France', lapangan_netral=False)

  Brazil  vs  Germany
PREDIKSI: Brazil MENANG

--- Alasan ---
Brazil punya win rate historis 66.1%, lebih tinggi 5.0 poin dibanding Germany (61.1%).
Tim dengan riwayat kemenangan lebih banyak di World Cup cenderung diprediksi menang lagi.

--- Probabilitas ---
Brazil Menang             46.2%
Germany Menang            35.5%
Seri                      18.3%

  Argentina  vs  France
PREDIKSI: Argentina MENANG

--- Alasan ---
Argentina punya win rate historis 53.4%, lebih tinggi 0.0 poin dibanding France (53.4%).
Tim dengan riwayat kemenangan lebih banyak di World Cup cenderung diprediksi menang lagi.
Faktor tambahan: Argentina bermain di kandang sendiri, memberi sedikit keuntungan ekstra.

--- Probabilitas ---
Argentina Menang          64.1%
Seri                      18.0%
France Menang             17.9%
